In [1]:
def main(datasources, start_date, end_date):
    """PIT quality + price behaviour + microstructure XGBoost factor.

    The model is trained only on a fixed historical interval.  The evaluator's
    injected tables and dates are used exclusively to construct prediction
    features, which prevents the public/test interval from leaking into model
    fitting.  The return value follows the BigAlpha contract exactly.
    """
    import numpy as np
    import pandas as pd
    import dai
    from xgboost import XGBRegressor

    # Use the complete official history, capped before the prediction window.
    # The fixed upper bound keeps the same model valid for the undisclosed
    # public window and the 2026 private window without querying beyond the
    # documented 2019-2024 historical dataset.
    prediction_start = pd.Timestamp(start_date).normalize()
    official_history_start = pd.Timestamp("2019-01-01")
    official_history_end = pd.Timestamp("2024-12-31")
    train_start_ts = official_history_start
    train_end_ts = min(
        prediction_start - pd.Timedelta(days=1), official_history_end
    )
    if train_end_ts < train_start_ts:
        raise ValueError("Prediction window starts before official training history.")
    train_start = train_start_ts.strftime("%Y-%m-%d")
    train_end = train_end_ts.strftime("%Y-%m-%d")
    train_bar_table = "bigalpha_2026_stock_bar1m"
    train_financial_table = "bigalpha_2026_financial"

    feature_columns = [
        "ret_1d",
        "gap",
        "intraday_return",
        "range_pct",
        "close_position",
        "book_imbalance",
        "relative_spread",
        "log_amount_20",
        "amount_surprise_20",
        "liquidity_nonlinear",
        "short_reversal_3",
        "momentum_60_excl_5",
        "volatility_10",
        "beta_60",
        "residual_volatility_20",
        "roe_proxy",
        "roa_proxy",
        "net_margin",
        "asset_turnover",
        "leverage",
        "revenue_growth_252",
        "profit_growth_252",
    ]

    def _normalise_date(frame):
        frame = frame.copy()
        frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
        # DAI may return dictionary/categorical encoding for instrument on some
        # tables and plain object strings on others.  Align the merge-key dtype.
        frame["instrument"] = frame["instrument"].astype(str)
        return frame

    def _query_daily_bars(table, query_start, query_end):
        # The five book levels are weighted toward prices nearest the touch.
        sql = f"""
        WITH minute_features AS (
            SELECT
                date,
                instrument,
                open,
                high,
                low,
                close,
                pre_close,
                volume,
                amount,
                (
                    5.0 * COALESCE(bid_volume1, 0)
                    + 4.0 * COALESCE(bid_volume2, 0)
                    + 3.0 * COALESCE(bid_volume3, 0)
                    + 2.0 * COALESCE(bid_volume4, 0)
                    + COALESCE(bid_volume5, 0)
                    - 5.0 * COALESCE(ask_volume1, 0)
                    - 4.0 * COALESCE(ask_volume2, 0)
                    - 3.0 * COALESCE(ask_volume3, 0)
                    - 2.0 * COALESCE(ask_volume4, 0)
                    - COALESCE(ask_volume5, 0)
                ) / (
                    5.0 * COALESCE(bid_volume1, 0)
                    + 4.0 * COALESCE(bid_volume2, 0)
                    + 3.0 * COALESCE(bid_volume3, 0)
                    + 2.0 * COALESCE(bid_volume4, 0)
                    + COALESCE(bid_volume5, 0)
                    + 5.0 * COALESCE(ask_volume1, 0)
                    + 4.0 * COALESCE(ask_volume2, 0)
                    + 3.0 * COALESCE(ask_volume3, 0)
                    + 2.0 * COALESCE(ask_volume4, 0)
                    + COALESCE(ask_volume5, 0)
                    + 1.0
                ) AS book_imbalance,
                (COALESCE(ask_price1, 0) - COALESCE(bid_price1, 0))
                / (0.5 * (COALESCE(ask_price1, 0) + COALESCE(bid_price1, 0))
                   + 1e-12) AS relative_spread
            FROM {table}
            WHERE date >= CAST('{query_start}' AS DATETIME)
              AND date < CAST('{query_end}' AS DATETIME) + INTERVAL 1 DAY
        )
        SELECT
            CAST(strftime(date, '%Y-%m-%d') AS DATETIME) AS date,
            instrument,
            ARG_MIN(open, date) AS open,
            MAX(high) AS high,
            MIN(low) AS low,
            ARG_MAX(close, date) AS close,
            ARG_MIN(pre_close, date) AS pre_close,
            MAX(volume) AS volume,
            MAX(amount) AS amount,
            AVG(book_imbalance) AS book_imbalance,
            AVG(relative_spread) AS relative_spread
        FROM minute_features
        GROUP BY instrument, strftime(date, '%Y-%m-%d')
        """
        return _normalise_date(
            dai.query(
                sql,
                # DAI interprets a bare end date as 00:00:00.  Using end-of-day
                # is essential: otherwise the cutoff run loses the cutoff
                # day's minute bars while the full-window run contains them.
                filters={"date": [query_start, f"{query_end} 23:59:59"]},
                compression=True,
            ).df()
        )

    def _query_financials(table, query_start, query_end):
        # category='lf', shift=0 is the official PIT-safe financial snapshot.
        sql = f"""
        SELECT
            CAST(date AS DATETIME) AS date,
            instrument,
            operating_revenue,
            net_profit_to_parent_shareholders,
            total_assets,
            total_equity_to_parent_shareholders
        FROM {table}
        WHERE category = 'lf' AND shift = 0
          AND date >= CAST('{query_start}' AS DATETIME)
          AND date < CAST('{query_end}' AS DATETIME) + INTERVAL 1 DAY
        """
        frame = dai.query(
            sql,
            filters={"date": [query_start, f"{query_end} 23:59:59"]},
            compression=True,
        ).df()
        frame = _normalise_date(frame)
        value_columns = [
            "operating_revenue",
            "net_profit_to_parent_shareholders",
            "total_assets",
            "total_equity_to_parent_shareholders",
        ]
        for column in value_columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        # A deterministic reduction is important because the look-ahead check
        # executes main() twice with different end_date values.
        return (
            frame.groupby(["date", "instrument"], as_index=False, sort=True)[value_columns]
            .max()
        )

    def _query_pool(query_start, query_end):
        pool = dai.query(
            f"""
            SELECT CAST(date AS DATETIME) AS date, instrument
            FROM bigalpha_2026_instruments
            WHERE date >= CAST('{query_start}' AS DATETIME)
              AND date < CAST('{query_end}' AS DATETIME) + INTERVAL 1 DAY
            """,
            filters={"date": [query_start, f"{query_end} 23:59:59"]},
            compression=True,
        ).df()
        return _normalise_date(pool).drop_duplicates(["date", "instrument"])

    def _safe_divide(numerator, denominator):
        denominator = denominator.where(denominator.abs() > 1e-12)
        return numerator / denominator

    def _build_features(bar_table, financial_table, period_start, period_end):
        period_start = pd.Timestamp(period_start).normalize()
        period_end = pd.Timestamp(period_end).normalize()
        # About 120 calendar days supplies at least 60 trading observations for
        # long momentum and beta features at prediction-window boundaries.
        bar_start = (period_start - pd.Timedelta(days=120)).strftime("%Y-%m-%d")
        fin_start = (period_start - pd.Timedelta(days=730)).strftime("%Y-%m-%d")
        end_text = period_end.strftime("%Y-%m-%d")

        bars = _query_daily_bars(bar_table, bar_start, end_text)
        pool = _query_pool(bar_start, end_text)
        financials = _query_financials(financial_table, fin_start, end_text)

        # Keep every stock-pool row.  Suspended or sparse stocks receive neutral
        # ranked features later, rather than silently reducing daily coverage.
        data = pool.merge(bars, on=["date", "instrument"], how="left")
        data = data.sort_values(["date", "instrument"]).reset_index(drop=True)
        financials = financials.sort_values(["date", "instrument"]).reset_index(drop=True)
        data = pd.merge_asof(
            data,
            financials,
            on="date",
            by="instrument",
            direction="backward",
            allow_exact_matches=True,
        )
        data = data.sort_values(["instrument", "date"]).reset_index(drop=True)

        numeric_columns = [
            "open", "high", "low", "close", "pre_close", "volume", "amount",
            "book_imbalance", "relative_spread", "operating_revenue",
            "net_profit_to_parent_shareholders", "total_assets",
            "total_equity_to_parent_shareholders",
        ]
        for column in numeric_columns:
            data[column] = pd.to_numeric(data[column], errors="coerce")

        data["ret_1d"] = _safe_divide(data["close"], data["pre_close"]) - 1.0
        data["gap"] = _safe_divide(data["open"], data["pre_close"]) - 1.0
        data["intraday_return"] = _safe_divide(data["close"], data["open"]) - 1.0
        data["range_pct"] = _safe_divide(data["high"], data["low"]) - 1.0
        data["close_position"] = (
            _safe_divide(data["close"] - data["low"], data["high"] - data["low"])
            - 0.5
        )
        data["log_amount"] = np.log1p(data["amount"].clip(lower=0))

        grouped = data.groupby("instrument", sort=False, group_keys=False)
        clipped_return = data["ret_1d"].clip(-0.30, 0.30)
        return_sum_3 = clipped_return.groupby(data["instrument"]).transform(
            lambda values: values.rolling(3, min_periods=2).sum()
        )
        return_sum_5 = clipped_return.groupby(data["instrument"]).transform(
            lambda values: values.rolling(5, min_periods=3).sum()
        )
        return_sum_60 = clipped_return.groupby(data["instrument"]).transform(
            lambda values: values.rolling(60, min_periods=30).sum()
        )
        # Separate horizons avoid the old exact duplication where reversal_5
        # was merely the negative of momentum_5.
        data["short_reversal_3"] = -return_sum_3
        data["momentum_60_excl_5"] = return_sum_60 - return_sum_5
        data["volatility_10"] = clipped_return.groupby(data["instrument"]).transform(
            lambda values: values.rolling(10, min_periods=5).std()
        )
        data["log_amount_20"] = grouped["log_amount"].transform(
            lambda values: values.rolling(20, min_periods=5).mean()
        )
        data["amount_surprise_20"] = data["log_amount"] - data["log_amount_20"]
        daily_amount_median = data.groupby("date")["log_amount"].transform("median")
        data["liquidity_nonlinear"] = (
            data["log_amount"] - daily_amount_median
        ).abs()

        # A same-day cross-sectional median return is the market proxy.  All
        # rolling beta and residual-volatility inputs use only dates up to t.
        data["market_return"] = data.groupby("date")["ret_1d"].transform("median")
        data["market_return"] = data["market_return"].clip(-0.20, 0.20)
        data["return_x_market"] = clipped_return * data["market_return"]
        data["market_return_sq"] = data["market_return"] ** 2
        rolling_mean_60 = lambda values: values.rolling(60, min_periods=30).mean()
        mean_return_60 = clipped_return.groupby(data["instrument"]).transform(
            rolling_mean_60
        )
        mean_market_60 = data["market_return"].groupby(data["instrument"]).transform(
            rolling_mean_60
        )
        mean_cross_60 = grouped["return_x_market"].transform(rolling_mean_60)
        mean_market_sq_60 = grouped["market_return_sq"].transform(rolling_mean_60)
        market_variance_60 = mean_market_sq_60 - mean_market_60**2
        data["beta_60"] = _safe_divide(
            mean_cross_60 - mean_return_60 * mean_market_60,
            market_variance_60,
        ).clip(-5.0, 5.0)
        residual_return = clipped_return - data["beta_60"] * data["market_return"]
        data["residual_volatility_20"] = residual_return.groupby(
            data["instrument"]
        ).transform(lambda values: values.rolling(20, min_periods=10).std())

        profit = data["net_profit_to_parent_shareholders"]
        revenue = data["operating_revenue"]
        assets = data["total_assets"]
        equity = data["total_equity_to_parent_shareholders"]
        data["roe_proxy"] = _safe_divide(profit, equity)
        data["roa_proxy"] = _safe_divide(profit, assets)
        data["net_margin"] = _safe_divide(profit, revenue)
        data["asset_turnover"] = _safe_divide(revenue, assets)
        data["leverage"] = _safe_divide(assets, equity)
        revenue_lag_252 = grouped["operating_revenue"].shift(252)
        profit_lag_252 = grouped["net_profit_to_parent_shareholders"].shift(252)
        data["revenue_growth_252"] = _safe_divide(
            revenue - revenue_lag_252,
            revenue.abs() + revenue_lag_252.abs(),
        )
        data["profit_growth_252"] = _safe_divide(
            profit - profit_lag_252,
            profit.abs() + profit_lag_252.abs(),
        )

        data = data[data["date"].between(period_start, period_end)].copy()
        data.replace([np.inf, -np.inf], np.nan, inplace=True)
        return data.sort_values(["date", "instrument"]).reset_index(drop=True)

    def _cross_sectional_rank(frame):
        ranked = frame.copy()
        for column in feature_columns:
            ranked[column] = ranked.groupby("date")[column].rank(pct=True) - 0.5
        ranked[feature_columns] = ranked[feature_columns].fillna(0.0)
        return ranked

    train = _build_features(
        train_bar_table, train_financial_table, train_start, train_end
    )
    # Build the next-trading-day label without a negative shift.  Each realised
    # return is moved back onto the preceding feature date using shift(+1) on
    # the date key.  Labels are used only inside the pre-start training window.
    labels = train[["date", "instrument", "ret_1d"]].copy()
    labels["date"] = labels.groupby("instrument", sort=False)["date"].shift(1)
    labels = labels.rename(columns={"ret_1d": "target"})
    labels = labels.dropna(subset=["date", "target"])
    train = train.merge(
        labels[["date", "instrument", "target"]],
        on=["date", "instrument"],
        how="inner",
        validate="one_to_one",
    )
    train["target"] = train.groupby("date")["target"].rank(pct=True) - 0.5
    train = _cross_sectional_rank(train)
    train = train.dropna(subset=["target"])
    train = train.sort_values(["date", "instrument"]).reset_index(drop=True)

    if len(train) < 1000:
        raise ValueError(
            "Insufficient strictly pre-start training observations; "
            "start_date must leave enough historical data."
        )

    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=80,
        max_depth=4,
        learning_rate=0.05,
        min_child_weight=50,
        subsample=1.0,
        colsample_bytree=1.0,
        reg_alpha=0.20,
        reg_lambda=5.0,
        tree_method="hist",
        # The official prefix-invariance check trains the model twice.  Single
        # threaded fitting removes parallel histogram reduction variability.
        n_jobs=1,
        random_state=42,
    )
    # All official history is retained, while a one-year half-life lets recent
    # regimes matter more.  Mean-one normalization keeps XGBoost's absolute
    # min_child_weight scale comparable with the previous model.
    age_days = (train_end_ts - train["date"]).dt.days.clip(lower=0)
    sample_weight = np.power(0.5, age_days / 365.0)
    sample_weight = sample_weight / sample_weight.mean()
    model.fit(
        train[feature_columns],
        train["target"],
        sample_weight=sample_weight,
    )

    test = _build_features(
        datasources["bar1m"], datasources["financial"], start_date, end_date
    )
    test = _cross_sectional_rank(test)
    test["factor"] = model.predict(test[feature_columns])
    # Ranking the score each day makes the submitted signal scale-stable and
    # preserves only the cross-sectional ordering used by RankIC evaluation.
    test["factor"] = test.groupby("date")["factor"].rank(pct=True) - 0.5
    test["factor"] = pd.to_numeric(test["factor"], errors="coerce")
    test["factor"] = test["factor"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    result = test[["date", "instrument", "factor"]].copy()
    result = result.drop_duplicates(["date", "instrument"], keep="last")
    return result.sort_values(["date", "instrument"]).reset_index(drop=True)
